# Build the Skolfie corpus into Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/build_corpus.ipynb)

This notebook runs roadmap step 2 in Google Colab and stores the result in your Google Drive:

1. mounts your Drive,
2. clones the Skolfie repo and installs it,
3. downloads HC3 and AuTexTification (Hugging Face) and M4GT (Google Drive),
4. builds `corpus.parquet`, `manifest.json` and `stats.md` into `My Drive/skolfie/data/processed/`.

Use **Runtime → Run all**. Approve the Drive access prompt when it appears. A CPU runtime is enough; the build takes a few minutes.

**Optional Colab secrets** (key icon in the left sidebar), only needed in these cases:
- `GITHUB_TOKEN`: if the repo is private (a fine-grained token with read access to `mohaaa825/Skolfie`).
- `HF_TOKEN`: if a Hugging Face download says the dataset is gated or rate-limited.

## 1. Settings

In [ ]:
# Where everything goes in your Drive.
DRIVE_ROOT = "/content/drive/MyDrive/skolfie/data"

# Keep the raw downloads in Drive too (~0.5-1 GB) so later rebuilds skip downloading.
# False keeps them on the Colab VM only (deleted when the runtime ends).
KEEP_RAW_IN_DRIVE = False

REPO = "mohaaa825/Skolfie"
BRANCH = "main"
M4GT_FOLDER = "https://drive.google.com/drive/folders/1xC-n5hHiXmaCLmA0q3p8Cbyd7Dk0dpDE"

# Build parameters (same defaults as scripts/build_corpus.py).
MIN_WORDS = 30
MAX_WORDS = 2000
BALANCE = "none"  # or "downsample"

## 2. Mount Google Drive

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

RAW_DIR = f"{DRIVE_ROOT}/raw" if KEEP_RAW_IN_DRIVE else "/content/skolfie_raw"
OUT_DIR = f"{DRIVE_ROOT}/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("raw files ->", RAW_DIR)
print("corpus    ->", OUT_DIR)

## 3. Clone and install Skolfie

In [ ]:
import subprocess


def secret(name):
    """Read a Colab secret, or return None if it is not set or not shared with this notebook."""
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if os.path.isdir("/content/Skolfie/.git"):
    subprocess.run(["git", "-C", "/content/Skolfie", "pull", "--ff-only"], check=True)
else:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie"]
    )
    if result.returncode != 0:
        raise SystemExit(
            "Clone failed. If the repo is private, add a GITHUB_TOKEN Colab secret and re-run."
        )

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[data]"

hf_token = secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## 4. Download M4GT from Google Drive

M4GT-Bench is only published as a shared Drive folder. Only the English subtask-A file is used
(`SubtaskA.jsonl`); `SubtaskA_split.jsonl` holds the same rows and is skipped to avoid duplicates.
If this cell fails (Drive sometimes rate-limits public folders), the build still runs without M4GT.

In [ ]:
import glob
import shutil

m4gt_dir = f"{RAW_DIR}/m4gt"
target = f"{m4gt_dir}/SubtaskA.jsonl"
os.makedirs(m4gt_dir, exist_ok=True)

if os.path.exists(target):
    print("M4GT already present:", target)
else:
    download_dir = "/content/m4gt_download"
    !gdown --folder "{M4GT_FOLDER}" -O "{download_dir}" --remaining-ok
    found = sorted(glob.glob(f"{download_dir}/**/*.jsonl", recursive=True))
    print("jsonl files in the M4GT folder:")
    for path in found:
        print(f"  {os.path.getsize(path) / 1e6:8.1f} MB  {os.path.relpath(path, download_dir)}")
    subtask_a = [p for p in found if os.path.basename(p).lower() == "subtaska.jsonl"]
    if subtask_a:
        shutil.copy(subtask_a[0], target)
        print("copied", subtask_a[0], "->", target)
    else:
        print("SubtaskA.jsonl not found; the build will continue without M4GT.")

## 5. Build the corpus

HC3 and AuTexTification are downloaded from Hugging Face by their loaders. A source that fails
is reported and skipped; the error message says which files it needed.

In [ ]:
!python scripts/build_corpus.py --raw-dir "{RAW_DIR}" --out-dir "{OUT_DIR}" \
    --min-words {MIN_WORDS} --max-words {MAX_WORDS} --balance {BALANCE}

## 6. Check the result

In [ ]:
import json

import pandas as pd

with open(f"{OUT_DIR}/manifest.json") as f:
    manifest = json.load(f)

print("corpus version:", manifest["corpus_version"])
print("records:       ", manifest["n_records"])
print("parquet size:  ", f"{os.path.getsize(f'{OUT_DIR}/corpus.parquet') / 1e6:.1f} MB")
print("errors:        ", manifest.get("errors") or "none")
print("warnings:      ", manifest.get("warnings") or "none")

corpus = pd.read_parquet(f"{OUT_DIR}/corpus.parquet")
display(pd.crosstab(corpus["domain"], corpus["label"], margins=True))
display(corpus.groupby(["source", "generator"]).size().rename("texts").to_frame())
display(corpus.sample(5, random_state=0)[["source", "label", "generator", "domain", "text"]])

The corpus is now in `My Drive/skolfie/data/processed/`. Train on it from Colab with:

```
!python scripts/train.py --data "{OUT_DIR}/corpus.parquet" --test-size 0.2
```

Paste the output of step 6 into the Skolfie project thread so the counts can be checked.